In [1]:
import torch
import model
import lancedb
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
from datafusion import functions as f
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import multiprocessing
import trackio
from tqdm.auto import tqdm

In [2]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])
    trackio.init(name=cfg.trackio.run_name, project=cfg.trackio.project, server_url=cfg.trackio.write_url)

* Trackio project initialized: ha_schmidhuber
* Trackio metrics will be sent to self-hosted server: https://trackio.flydexo.com


* Apple Silicon detected, enabling automatic GPU/system metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: <10k


In [3]:
vae = model.AutoEncoder(cfg).to(cfg.device)

In [4]:
class EpisodicDataset(torch.utils.data.Dataset):
    def __init__(self, cfg):
        super().__init__()
        self.lancedb_uri = cfg.dataset.lancedb_uri
        self.img_size = cfg.dataset.img_size
        self._table = None
        self._len = lancedb.connect(cfg.dataset.lancedb_uri).open_table("episodes").count_rows()

    def _get_table(self):
        if self._table is None:
            self._table = lancedb.connect(self.lancedb_uri).open_table("episodes")
        return self._table

    def __getitem__(self, idx):
        episode = self._get_table().search().where(f'episode_id = {idx}').limit(1).to_arrow()
        frames_np = np.array(episode.column('observations').combine_chunks()[0].as_py(), dtype=np.uint8)
        frames_np = frames_np.reshape(-1, self.img_size, self.img_size, 3)
        t = torch.from_numpy(frames_np).clone().permute(0, 3, 1, 2).float() / 255.0
        return t[:cfg.dataset.max_episode_steps + 1]  # (T, 3, H, W)

    def __len__(self):
        return self._len

In [5]:
ds = EpisodicDataset(cfg)
train_ds, val_ds = random_split(ds, [0.9, 0.1])
print(f"{len(train_ds)} train episodes | {len(val_ds)} val episodes")

5130 train episodes | 570 val episodes


In [6]:
# Cat all frames from N episodes into one big batch — ~8x more GPU memory used per step
def collate_episodes(batch):
    return torch.cat(batch, dim=0)  # (sum(T_i), 3, H, W)

BATCH_SIZE = 4  # episodes per step; 8 × ~300 frames ≈ 12 GB GPU memory
num_workers = min(6, multiprocessing.cpu_count() - 1)
_kwargs = dict(
    batch_size=BATCH_SIZE, collate_fn=collate_episodes,
    num_workers=num_workers, persistent_workers=num_workers > 0,
    prefetch_factor=4 if num_workers > 0 else None,
    multiprocessing_context='fork' if num_workers > 0 else None,
)
train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)

In [7]:
img_size = cfg.dataset.img_size
optimizer = torch.optim.AdamW(vae.parameters(), cfg.training.vae.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.training.vae.nb_epochs)

for epoch in tqdm(range(cfg.training.vae.nb_epochs), desc="Epochs"):
    vae.train()
    for x in train_loader:
        x = x.to(cfg.device)  # (B*T, 3, H, W) float32, already normalized
        x_recon, kl = vae(x)
        loss = F.mse_loss(x_recon, x) + cfg.vae.beta * kl
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        trackio.log({"training-loss": loss.item(), "training-kl": kl.item()})

    vae.eval()
    with torch.no_grad():
        for x in val_loader:
            x = x.to(cfg.device)
            x_recon, kl = vae(x)
            loss = F.mse_loss(x_recon, x) + cfg.vae.beta * kl
            trackio.log({"val-loss": loss.item(), "val-kl": kl.item()})
    scheduler.step()

Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/

In [8]:
#trackio.finish()

In [9]:
torch.save(vae.state_dict(), f'./vae-{cfg.trackio.run_name}.pt') 